<a href="https://colab.research.google.com/github/catSushiRoll/BSProjek/blob/main/MaLeBSP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UAS TPIB 2024/2025 (Bagian B)

Contains CNN program to classify epileptic based on EEG signals
<br>Link to dataset : https://www.ukbonn.de/epileptologie/arbeitsgruppen/ag-lehnertz-neurophysik/downloads/#:~:text=



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
%pip install tensorflow

### Conversion File to CSV
This code is to make a CSV version of the dataset given

In [ ]:
import os
import numpy as np
import pandas as pd

def read_signal_from_txt(file_path):
    with open(file_path, 'r') as f:
        data = f.readlines()
    data = [float(line.strip()) for line in data]
    return data

def load_dataset_from_folder(folder_path, label):
    rows = []
    for filename in sorted(os.listdir(folder_path)):
        if filename.endswith('.txt'):
            file_path = os.path.join(folder_path, filename)
            signal = read_signal_from_txt(file_path)
            if len(signal) == 4097 or len(signal) == 4098: 
                if signal:
                    rows.append([label] + signal)
                else:
                    print(f"Warning: File {filename} is empty or contains no valid float data.")
    return rows

base_path = "/content/drive/MyDrive/dataset eeg"

categories = {
    'Z': 0, # non-epileptic
    'O': 0, # non-epileptic
    'N': 0, # non-epileptic
    'F': 0, # non-epileptic
    'S': 1   # epileptic
}

all_data = []

for folder, label in categories.items():
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        data = load_dataset_from_folder(folder_path, label)
        all_data.extend(data)
    else:
        print(f"Warning: Directory not found: {folder_path}")

df = pd.DataFrame(all_data)

if not df.empty:
    output_path = os.path.join('/content/drive/MyDrive/dataset eeg', 'EEG_epilepsy_dataset.csv')
    num_data_cols = df.shape[1] - 1 if df.shape[1] > 0 else 0
    header = ["label"] + [f"v{i}" for i in range(1, num_data_cols + 1)]

    if len(header) == df.shape[1]:
        df.to_csv(output_path, index=False, header=header)
        print(f"Conversion done. Datasets saved in {output_path}")
    else:
        print(f"Error: Header length ({len(header)}) does not match DataFrame columns ({df.shape[1]}). Cannot save CSV.")

else:
    print("No data was loaded from the specified folders. CSV not saved.")

No data was loaded from the specified folders. CSV not saved.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

df = pd.read_csv("/content/drive/MyDrive/dataset eeg/EEG_epilepsy_dataset.csv")

X = df.drop('label', axis=1).values  # shape: (200, 4097)
y = df['label'].values               # shape: (200,)

X = X / np.max(np.abs(X))

X = X.reshape((X.shape[0], X.shape[1], 1))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

model = Sequential()
model.add(Conv1D(32, kernel_size=7, activation='relu', input_shape=(4097, 1)))
model.add(MaxPooling1D(pool_size=2))
model.add(Conv1D(64, kernel_size=5, activation='relu'))
model.add(MaxPooling1D(pool_size=2))
model.add(Flatten())
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_final, y_train_final,
    epochs=20,
    batch_size=16,
    validation_data=(X_val, y_val)
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Akurasi pada data uji: {acc:.2f}")

plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Akurasi Model')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid(True)
plt.legend()
plt.show()

ModuleNotFoundError: No module named 'tensorflow'